# 0.0 Imports

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import sklearn.metrics as mt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.pipeline import Pipeline

# 0.1 - Load Datasets

In [14]:
actual_folder = Path.cwd()
print(actual_folder)

main_folder = actual_folder.parent.parent
print(main_folder)

path_folder_dataset = main_folder / 'dataset' / 'regression_datasets'

if path_folder_dataset.exists():
    print(f'Caminho existente: {path_folder_dataset}')
else:
    print(f'Erro: O arquivo não foi encontrado {path_folder_dataset}')

d:\repos\projetos\ml_trials_algorithm\notebooks\regressao
d:\repos\projetos\ml_trials_algorithm
Caminho existente: d:\repos\projetos\ml_trials_algorithm\dataset\regression_datasets


In [15]:
#Carrega os datasets da pasta - Treinamento / Validação e Teste

dataset_traning_X = path_folder_dataset / 'a_traninig' / 'X_training.csv'
dataset_traning_y = path_folder_dataset / 'a_traninig' / 'y_training.csv'

dataset_validation_X = path_folder_dataset / 'b_validation' / 'X_validation.csv'
dataset_validation_y = path_folder_dataset / 'b_validation' / 'y_validation.csv'

dataset_test_X = path_folder_dataset / 'c_test' / 'X_test.csv'
dataset_test_y = path_folder_dataset / 'c_test' / 'y_test.csv'

X_train = pd.read_csv(dataset_traning_X)
y_train = pd.read_csv(dataset_traning_y)

X_val = pd.read_csv(dataset_validation_X)
y_val = pd.read_csv(dataset_validation_y)

X_test = pd.read_csv(dataset_test_X)
y_test = pd.read_csv(dataset_test_y)

# 1.0 - Training Algorithm

## Passo 1 — Treino com parâmetros default

In [16]:
# Instanciar o modelo com parâmetros default (grau 1)
model_def = Pipeline([
    ('features', PolynomialFeatures(degree=1)),
    ('model', Ridge())
])

# Treinar com dados de treino
model_def.fit(X_train, y_train.values.ravel())

# Predições no treino e na validação
yhat_train_def = model_def.predict(X_train)
yhat_val_def   = model_def.predict(X_val)

## Passo 2 — Performance no treino (default)

In [17]:
# Métricas no conjunto de TREINO com parâmetros default
r2_train_def   = mt.r2_score(y_train, yhat_train_def)
mse_train_def  = mt.mean_squared_error(y_train, yhat_train_def)
rmse_train_def = np.sqrt(mse_train_def)
mae_train_def  = mt.mean_absolute_error(y_train, yhat_train_def)
mape_train_def = mt.mean_absolute_percentage_error(y_train, yhat_train_def)

print("--- Performance no Treino (Default) ---")
print(f"R²:   {r2_train_def:.4f}")
print(f"MSE:  {mse_train_def:.2f}")
print(f"RMSE: {rmse_train_def:.2f}")
print(f"MAE:  {mae_train_def:.2f}")
print(f"MAPE: {mape_train_def * 100:.2f}%")

--- Performance no Treino (Default) ---
R²:   0.0461
MSE:  456.00
RMSE: 21.35
MAE:  17.00
MAPE: 865.34%


## Passo 3 — Performance na validação (default)

In [18]:
# Métricas no conjunto de VALIDAÇÃO com parâmetros default
r2_val_def   = mt.r2_score(y_val, yhat_val_def)
mse_val_def  = mt.mean_squared_error(y_val, yhat_val_def)
rmse_val_def = np.sqrt(mse_val_def)
mae_val_def  = mt.mean_absolute_error(y_val, yhat_val_def)
mape_val_def = mt.mean_absolute_percentage_error(y_val, yhat_val_def)

print("--- Performance na Validação (Default) ---")
print(f"R²:   {r2_val_def:.4f}")
print(f"MSE:  {mse_val_def:.2f}")
print(f"RMSE: {rmse_val_def:.2f}")
print(f"MAE:  {mae_val_def:.2f}")
print(f"MAPE: {mape_val_def * 100:.2f}%")

--- Performance na Validação (Default) ---
R²:   0.0399
MSE:  458.45
RMSE: 21.41
MAE:  17.04
MAPE: 868.24%


## Passo 4 — Ajuste de hiperparâmetros

In [19]:
print("--- Testando Regressão Polinomial (Grau 2) com Penalização Ridge ---")
melhor_alpha_ridge = 1.0
melhor_r2_ridge = -999

# Alphas controlados para o Ridge
list_alpha_ridge = [0.01, 0.1, 1.0, 10.0, 100.0]

for alpha in list_alpha_ridge:
    # Cria o pipeline composto para o Ridge
    model_poly_ridge = Pipeline([
        ('features_polinomiais', PolynomialFeatures(degree=2, include_bias=False)),
        ('regressor_ridge', Ridge(alpha=alpha, solver='cholesky', random_state=42))
    ])
    
    try:
        model_poly_ridge.fit(X_train, y_train.values.ravel())
        yhat_train = model_poly_ridge.predict(X_train)
        yhat_val = model_poly_ridge.predict(X_val)
        
        r2_train = mt.r2_score(y_train, yhat_train)
        r2_val = mt.r2_score(y_val, yhat_val)
        rmse_val = np.sqrt(mt.mean_squared_error(y_val, yhat_val))
        
        print(f"Grau: 2 | Alpha: {alpha:<6} | R² Treino: {r2_train:.4f} | R² Val: {r2_val:.4f} | RMSE Val: {rmse_val:.2f}")
        
        if r2_val > melhor_r2_ridge:
            melhor_r2_ridge = r2_val
            melhor_alpha_ridge = alpha
            
    except Exception as e:
        print(f"Erro no Ridge Alpha {alpha}: {e}")

print("=" * 85)
print(f"-> VENCEDOR DO ENSAIO POLINOMIAL + RIDGE:")
print(f"Melhor Alpha (Ridge): {melhor_alpha_ridge}")
print(f"Maior R² obtido na Validação: {melhor_r2_ridge:.4f}\n")

--- Testando Regressão Polinomial (Grau 2) com Penalização Ridge ---
Grau: 2 | Alpha: 0.01   | R² Treino: 0.0942 | R² Val: 0.0666 | RMSE Val: 21.11
Grau: 2 | Alpha: 0.1    | R² Treino: 0.0941 | R² Val: 0.0672 | RMSE Val: 21.10
Grau: 2 | Alpha: 1.0    | R² Treino: 0.0932 | R² Val: 0.0677 | RMSE Val: 21.10
Grau: 2 | Alpha: 10.0   | R² Treino: 0.0893 | R² Val: 0.0672 | RMSE Val: 21.10
Grau: 2 | Alpha: 100.0  | R² Treino: 0.0792 | R² Val: 0.0636 | RMSE Val: 21.15
-> VENCEDOR DO ENSAIO POLINOMIAL + RIDGE:
Melhor Alpha (Ridge): 1.0
Maior R² obtido na Validação: 0.0677



## Passo 5 — Performance com modelo tunado (treino e validação)

In [20]:
# Modelo com os melhores hiperparâmetros encontrados, treinado apenas em X_train
model_tunado = Pipeline([
    ('features_polinomiais', PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor_ridge', Ridge(alpha=melhor_alpha_ridge, solver='cholesky', random_state=42))
])
model_tunado.fit(X_train, y_train.values.ravel())

yhat_train_tunado = model_tunado.predict(X_train)
yhat_val_tunado   = model_tunado.predict(X_val)

r2_train_tunado   = mt.r2_score(y_train, yhat_train_tunado)
mse_train_tunado  = mt.mean_squared_error(y_train, yhat_train_tunado)
rmse_train_tunado = np.sqrt(mse_train_tunado)
mae_train_tunado  = mt.mean_absolute_error(y_train, yhat_train_tunado)
mape_train_tunado = mt.mean_absolute_percentage_error(y_train, yhat_train_tunado)

r2_val_tunado   = mt.r2_score(y_val, yhat_val_tunado)
mse_val_tunado  = mt.mean_squared_error(y_val, yhat_val_tunado)
rmse_val_tunado = np.sqrt(mse_val_tunado)
mae_val_tunado  = mt.mean_absolute_error(y_val, yhat_val_tunado)
mape_val_tunado = mt.mean_absolute_percentage_error(y_val, yhat_val_tunado)

print("--- Performance com Modelo Tunado ---")
print(f"Treino    → R²: {r2_train_tunado:.4f} | RMSE: {rmse_train_tunado:.2f} | MAE: {mae_train_tunado:.2f} | MAPE: {mape_train_tunado*100:.2f}%")
print(f"Validação → R²: {r2_val_tunado:.4f} | RMSE: {rmse_val_tunado:.2f} | MAE: {mae_val_tunado:.2f} | MAPE: {mape_val_tunado*100:.2f}%")

--- Performance com Modelo Tunado ---
Treino    → R²: 0.0932 | RMSE: 20.82 | MAE: 16.47 | MAPE: 837.27%
Validação → R²: 0.0677 | RMSE: 21.10 | MAE: 16.74 | MAPE: 856.90%


## Passo 6 — União treino + validação

In [21]:
# Unir treino + validação para formar o conjunto final de treinamento
X_train_final = pd.concat([X_train, X_val])
y_train_final = pd.concat([y_train, y_val])

print(f"X_train_final shape: {X_train_final.shape}")
print(f"y_train_final shape: {y_train_final.shape}")

X_train_final shape: (15068, 13)
y_train_final shape: (15068, 1)


## Passo 7 — Retreino com melhores parâmetros

In [22]:
# Retreino com os melhores hiperparâmetros sobre o conjunto final (treino + validação)
model_final = Pipeline([
    ('features_polinomiais', PolynomialFeatures(degree=2, include_bias=False)),
    ('regressor_ridge', Ridge(alpha=melhor_alpha_ridge, solver='cholesky', random_state=42))
])
model_final.fit(X_train_final, y_train_final.values.ravel())

,steps,"[('features_polinomiais', ...), ('regressor_ridge', ...)]"
,transform_input,None
,memory,None
,verbose,False
,degree,2
,interaction_only,False
,include_bias,False
,order,'C'
,alpha,1.0
,fit_intercept,True
,copy_X,True


## Passo 8 — Performance no teste

In [23]:
# Predição no conjunto de teste (única vez que X_test é tocado)
yhat_test = model_final.predict(X_test)

# Métricas no conjunto de TESTE
r2_test   = mt.r2_score(y_test, yhat_test)
mse_test  = mt.mean_squared_error(y_test, yhat_test)
rmse_test = np.sqrt(mse_test)
mae_test  = mt.mean_absolute_error(y_test, yhat_test)
mape_test = mt.mean_absolute_percentage_error(y_test, yhat_test)

print("--- Performance no Teste (Final) ---")
print(f"R²:   {r2_test:.4f}")
print(f"MSE:  {mse_test:.2f}")
print(f"RMSE: {rmse_test:.2f}")
print(f"MAE:  {mae_test:.2f}")
print(f"MAPE: {mape_test * 100:.2f}%")

--- Performance no Teste (Final) ---
R²:   0.0902
MSE:  442.97
RMSE: 21.05
MAE:  16.74
MAPE: 830.85%


## Passo 9 — Quadro Comparativo — Diagnóstico Completo

In [24]:
data_comparativo = {
    'Conjunto': ['Treino (Default)', 'Validação (Default)', 'Treino (Tunado)', 'Validação (Tunado)', 'Teste (Final)'],
    'R²':    [r2_train_def,   r2_val_def,   r2_train_tunado,   r2_val_tunado,   r2_test],
    'RMSE':  [rmse_train_def, rmse_val_def, rmse_train_tunado, rmse_val_tunado, rmse_test],
    'MAE':   [mae_train_def,  mae_val_def,  mae_train_tunado,  mae_val_tunado,  mae_test],
    'MAPE':  [f'{mape_train_def*100:.2f}%', f'{mape_val_def*100:.2f}%',
              f'{mape_train_tunado*100:.2f}%', f'{mape_val_tunado*100:.2f}%',
              f'{mape_test*100:.2f}%'],
}
df_comparativo = pd.DataFrame(data_comparativo)
print("\n--- Quadro Comparativo — Diagnóstico Completo ---")
print(df_comparativo.to_string(index=False))


--- Quadro Comparativo — Diagnóstico Completo ---
           Conjunto       R²      RMSE       MAE    MAPE
   Treino (Default) 0.046058 21.354072 16.998308 865.34%
Validação (Default) 0.039928 21.411340 17.039472 868.24%
    Treino (Tunado) 0.093171 20.820073 16.471972 837.27%
 Validação (Tunado) 0.067699 21.099394 16.738741 856.90%
      Teste (Final) 0.090231 21.046790 16.742214 830.85%
